# 05 · Differential splicing (rMATS)

A cassette exon is an exon that cells sometimes include in the mRNA and sometimes skip. PSI (percent spliced in) is the fraction of transcripts that include it, from 0 to 1. rMATS counts the RNA-seq reads that cross exon junctions, both those that include the exon and those that skip it. From these it computes PSI per replicate and tests whether PSI differs between knockdown and control. ΔPSI is PSI after knockdown minus PSI in control.

This notebook runs rMATS on notebook 04's BAMs. It then sorts the exons into the three groups that the splice maps compare and moves them from hg19 to GRCh38, the genome version RBP-ARK uses.

On Google Colab, run the next cell first in every session. It copies this repository onto the Colab machine and, with `SAVE_TO_DRIVE = True`, saves results and the figure to `My Drive/CS3_workflow` (Colab asks for permission to connect your Drive). `LITE = True` uses a small subset of the data that fits free Colab. On your own computer the cell does nothing.

In [ ]:
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess
    REPO = "https://github.com/yeolab/encode-rbp-case-study"
    CLONE = "/content/encode-rbp-case-study"
    LITE = True    # True: small subset of the data, fits free Colab (about 1 h). False: all the data (5-8 h, ~50 GB)
    SAVE_TO_DRIVE = True    # results and the figure in My Drive/CS3_workflow; False: on this Colab machine only
    if not os.path.isdir(CLONE):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, CLONE], check=True)
    WORKFLOW = CLONE + "/notebooks/workflow"
    OUTPUT = None
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT = "/content/drive/MyDrive/CS3_workflow"
    sys.path.insert(0, WORKFLOW)
    import colab_setup
    colab_setup.setup(WORKFLOW, lite=LITE, output_folder=OUTPUT, rmats=True)

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

sys.path.insert(0, ".")
import workflow_utils as U

ROOT, DATA = U.folders()
RESULTS = ROOT / "results" / "workflow"
API_CACHE = DATA / "downloads" / "rbpark_api"
print("case study folder:", ROOT)
print("data folder:", DATA)
import os
import shutil
import subprocess

BAM_DIR = DATA / "downloads" / "shrna_bams_hg19"
REF_DIR = DATA / "downloads" / "encode_reference"
GENOME_CACHE = DATA / "downloads" / "genome"
DIRS = U.data_dirs(DATA)
EVENT_DIR = DIRS["events"]
THREADS = os.cpu_count() or 4

## Run rMATS

rMATS needs gene models in the BAMs' genome version, so it uses GENCODE v40 mapped to hg19 (`gencode.v40lift37`). The reads are paired-end, 100 nt, from strand-specific libraries (`--libType fr-firststrand`). One run takes 30 to 60 minutes. A finished run with the same settings is reused.

In lite mode rMATS runs on the PICALM-only BAMs with PICALM's gene models, which takes a minute. It tests only PICALM's exons, so the splice maps further down use the genome-wide results of the full run, which come with `lite_data/`.

In [ ]:
GTF = U.LITE_DIR / "gencode.v40lift37.PICALM.gtf" if U.LITE else REF_DIR / "gencode.v40lift37.annotation.gtf"
if not GTF.exists():
    U.fetch(U.GENCODE_V40_LIFT37, str(GTF) + ".gz")
    subprocess.check_call(["gunzip", "-f", str(GTF) + ".gz"])

# rMATS runs with this kernel's Python (or, on Colab, from its own environment: RMATS_CMD, set by colab_setup)
RMATS = os.environ["RMATS_CMD"].split() if "RMATS_CMD" in os.environ else [sys.executable, os.path.realpath(shutil.which("rmats.py"))]
version = subprocess.run(RMATS + ["--version"], capture_output=True, text=True).stdout.strip()
settings = "annotation {}\nlibType {}\nrMATS {}".format(GTF.name, U.RMATS_SETTINGS["lib_type"], version)
print(settings)

manifest = pd.read_csv(BAM_DIR / ("manifest_qki_lite.tsv" if U.LITE else "manifest_qki.tsv"), sep="\t")
se_tables = {}
for key in U.KNOCKDOWNS:
    print(key, "running rMATS..." if not (DIRS["rmats_run"] / key / "SE.MATS.JC.txt").exists() else "", flush=True)
    se_tables[key], status = U.run_rmats(key, manifest, GTF, DIRS["rmats_run"], THREADS, RMATS, settings)
    print(key, status, se_tables[key])

In lite mode, the next cell compares the PICALM rows of this small run with the full run. Junction reads, PSI and ΔPSI match: rMATS counts each exon's reads on their own. The FDR differs, because it corrects for the number of exons tested, here a handful instead of tens of thousands. The rest of the workflow uses the full run's table.

In [ ]:
if U.LITE:
    rows = []
    for key in U.KNOCKDOWNS:
        for exon, info in U.EXONS.items():
            for source, path in (("lite run (PICALM only)", se_tables[key]), ("full run", DIRS["rmats"] / key / "SE.MATS.JC.txt")):
                st = U.exon_stats(U.exon_row(path, exon))
                rows.append({"knockdown": key, "exon": info["name"], "source": source, "inclusion reads (KD)": st["IJC_kd"],
                             "skipping reads (KD)": st["SJC_kd"], "PSI KD": st["psi_kd"], "dPSI": st["dpsi"],
                             "P": st["pvalue"], "FDR": st["fdr"]})
    display(pd.DataFrame(rows))
    se_tables = {key: DIRS["rmats"] / key / "SE.MATS.JC.txt" for key in U.KNOCKDOWNS}

## Three groups of exons

| Group | Rule |
| --- | --- |
| included upon knockdown | FDR < 0.1, P < 0.05, ΔPSI > 0.05 |
| excluded upon knockdown | FDR < 0.1, P < 0.05, ΔPSI < −0.05 |
| background | P > 0.5, PSI between 0.05 and 0.95 in both conditions |

An exon included more after knockdown is one QKI normally represses; one excluded more is one QKI normally promotes. Background exons are alternative but did not change.

Exons of the genes in `U.EXCLUDED_GENES` (one Ensembl gene ID) are left out of all three groups.

rMATS can report one exon several times, for example with two different 3′ splice sites. Each group keeps one event per exon by rbp-maps' rule: the form with the most inclusion reads. When two forms tie, the one listed first in the rMATS table is kept, so every computer keeps the same events. Then all three exons of each event (the cassette exon and its two neighbours) are converted from hg19 to GRCh38 with UCSC's chain file, and events that do not convert cleanly are dropped.

In [ ]:
rows = []
for key in U.KNOCKDOWNS:
    counts, lists = U.build_event_lists(key, se_tables[key], EVENT_DIR, GENOME_CACHE)
    for set_name, n in counts.items():
        rows.append({"knockdown": key, "group": set_name, **n})
    for exon, info in U.EXONS.items():
        print(key, info["name"], "->", U.which_set(lists, exon))
event_counts = pd.DataFrame(rows)
event_counts.to_csv(RESULTS / "05_event_counts.tsv", sep="\t", index=False)
event_counts

Expected: in both knockdowns the 150-nt exon is in the included group and the 24-nt exon in the excluded group. For HepG2, about 1,500 included, 900 excluded and 2,300 background exons remain.